In [1]:
# Import necessary libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import TensorDataset, DataLoader,Dataset
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import albumentations as albu
import albumentations

from fastprogress import master_bar, progress_bar
import gc

import random
from tqdm import tqdm

from copy import deepcopy
import joblib
from sklearn.model_selection import KFold
from lightgbm import LGBMClassifier as lgbm

import timm

from sklearn.metrics import f1_score
from scipy.stats import kurtosis, skew
from xgboost import XGBClassifier as xgb

In [2]:
CALC_OOF=False
bestthresh=0.49 # if CALC_OOF=True this will be re-calculated, reseted to 0.49

# Set the path of the trained models and model weights
TRAINED_MODELS_PATH='/kaggle/input/landslide-class-weights/'

# train data needed if OOF is to be calculated
# Define paths for the dataset (remember to unzip the dataset first!)
data_path='/kaggle/input/slideandseekclasificationlandslidedetectiondataset/'
train_csv_path = data_path+'Train.csv'  # Path to the training labels CSV file
test_csv_path = data_path+'Test.csv'    # Path to the test image IDs CSV file
train_data_path = data_path+'train_data/train_data'  # Folder where .npy train files are located
test_data_path = data_path+'test_data/test_data'    # Folder where .npy test files are located

In [3]:
# Params used - don't edit
N_folds=5
THRESH=0.5
num_workers=4
BA=1
BS=16
if CALC_OOF==False:
    bestthresh=0.49 

# version_params
experimentation=pd.DataFrame({
    'Version': {0: 'v18b',  1: 'v46',  2: 'v44',  3: 'v39',  4: 'v37',  5: 'v33',  6: 'v31',   8: 'v20',  9: 'v10'},
    'OOF T0.5': {0: 0.874,  1: 0.8943,  2: 0.8708,  3: 0.8986,  4: 0.9003,  5: 0.8963,  6: 0.8976,  8: 0.8496,  9: 0.8744},
    'OOF Tbest': {0: 0.8762,  1: 0.8991,  2: 0.8748,  3: 0.9009,  4: 0.9051,  5: 0.9026,  6: 0.8999,   8: 0.852,  9: 0.8751},
    'Subm T0.5': {0: 0.8792,  1: 0.91,  2: 0.8786,  3: 0.9082,  4: 0.9136,  5: 0.9117,  6: 0.9148,    8: 0.8475,  9: 0.9095},
    'Channels': {0: '0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17',  1: '0,1,2,3, 4,5,8,9, 6,7,10,11',  2: '0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17',
                 3: '0,1,2,3, 6,7,10,11, 12,13,14,15,16,17',  4: '0,1,2,3, 6,7,10,11, 12,13,14,15,16,17',  5: '0,1,2,3, 6,7,10,11',
                 6: '0,1,2,3, 6,7,10,11',    8: '4,5,6,7,8,9,10,11,12,13,14,15,16,17',  9: '0,1,2,3'},
    'Epochs': {0: np.nan,  1: 20.0,  2: np.nan,  3: 10.0,  4: 15.0,  5: 20.0,  6: 20.0,  8: 20.0,  9: 20.0},
    'Size': {0: 64,  1: 256,  2: 64,  3: 256,  4: 256,  5: 256,  6: 256,   8: 256,  9: 256},
    'Scale': {0: np.nan,  1: 'standardScale',  2: np.nan,  3: 'standardScale',  4: 'standardScale',  5: 'robust 05,50,95',  6: 'standardScale',
              8: 'robust 05,50,95',  9: 'robust 05,50,95'},
    'Model': {0: 'LGBM',  1: 'combinemodelsV3c',  2: 'XGB',  3: 'combinemodelsV3a',  4: 'combinemodelsV3a',  5: 'combinemodelsV2a',
              6: 'combinemodelsV2a',  8: 'timm',  9: 'timm'},
    'Encoder': {0: '18channels*7statistics=126 features',  1: 'vit_medium_patch16_rope_reg1_gap_256.sbb_in1k',  2: '18channels*7statistics=126 features',
                3: 'caformer_s18.sail_in22k_ft_in1k',  4: 'vit_pwee_patch16_reg1_gap_256.sbb_in1k',  5: 'vit_pwee_patch16_reg1_gap_256.sbb_in1k',
                6: 'vit_pwee_patch16_reg1_gap_256.sbb_in1k',  8: 'vit_medium_patch16_reg4_gap_256.sbb_in12k_ft_in1k',
                9: 'maxvit_rmlp_tiny_rw_256.sw_in1k'},
    'Notes': {0: 'feature eng. from optical bands, RERUN',  1: np.nan,  2: 'feature eng. from optical bands',  3: 'feature eng. from optical bands',
              4: 'feature eng. from optical bands',  5: np.nan,  6: 'rerun over previous v31, renamed old weights to v31a except fold0',
              8: 'feature eng. from optical bands',  9: np.nan}}).reset_index(drop=True)
experimentation

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,Version,OOF T0.5,OOF Tbest,Subm T0.5,Channels,Epochs,Size,Scale,Model,Encoder,Notes
0,v18b,0.8740,0.8762,0.8792,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17",NaN,64,NaN,LGBM,18channels*7statistics=126 features,"feature eng. from optical bands, RERUN"
1,v46,0.8943,0.8991,0.9100,"0,1,2,3, 4,5,8,9, 6,7,10,11",20.0,256,standardScale,combinemodelsV3c,vit_medium_patch16_rope_reg1_gap_256.sbb_in1k,NaN
2,v44,0.8708,0.8748,0.8786,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17",NaN,64,NaN,XGB,18channels*7statistics=126 features,feature eng. from optical bands
3,v39,0.8986,0.9009,0.9082,"0,1,2,3, 6,7,10,11, 12,13,14,15,16,17",10.0,256,standardScale,combinemodelsV3a,caformer_s18.sail_in22k_ft_in1k,feature eng. from optical bands
4,v37,0.9003,0.9051,0.9136,"0,1,2,3, 6,7,10,11, 12,13,14,15,16,17",15.0,256,standardScale,combinemodelsV3a,vit_pwee_patch16_reg1_gap_256.sbb_in1k,feature eng. from optical bands
5,v33,0.8963,0.9026,0.9117,"0,1,2,3, 6,7,10,11",20.0,256,"robust 05,50,95",combinemodelsV2a,vit_pwee_patch16_reg1_gap_256.sbb_in1k,NaN
6,v31,0.8976,0.8999,0.9148,"0,1,2,3, 6,7,10,11",20.0,256,standardScale,combinemodelsV2a,vit_pwee_patch16_reg1_gap_256.sbb_in1k,"rerun over previous v31, renamed old weights t..."
7,v20,0.8496,0.8520,0.8475,"4,5,6,7,8,9,10,11,12,13,14,15,16,17",20.0,256,"robust 05,50,95",timm,vit_medium_patch16_reg4_gap_256.sbb_in12k_ft_in1k,feature eng. from optical bands
8,v10,0.8744,0.8751,0.9095,"0,1,2,3",20.0,256,"robust 05,50,95",timm,maxvit_rmlp_tiny_rw_256.sw_in1k,NaN


In [4]:
# Functions and Classes
class SmoothF1Loss(nn.Module):
    def __init__(self, epsilon=1e-7):
        super(SmoothF1Loss, self).__init__()
        self.epsilon = epsilon

    def forward(self, logits, targets):
        """
        Computes a smooth F1 loss.

        Args:
            logits: Predicted raw outputs (before sigmoid), shape (batch_size,)
            targets: Ground truth binary labels, shape (batch_size,)

        Returns:
            Loss value to minimize (1 - soft F1 score)
        """
        # Apply sigmoid to convert logits to probabilities
        probs = torch.sigmoid(logits)

        # Ensure targets are float for arithmetic
        targets = targets.float()

        # True Positives, False Positives, False Negatives (soft)
        tp = (probs * targets).sum()
        fp = (probs * (1 - targets)).sum()
        fn = ((1 - probs) * targets).sum()

        # Smooth F1 calculation
        f1 = (2 * tp + self.epsilon) / (2 * tp + fp + fn + self.epsilon)

        # Convert to loss (1 - F1 to maximize F1)
        return 1 - f1
def find_opt_thresh(targets, preds):
    bestthresh=0.5
    bestscore=-np.inf
    for i in range(98):
        thresh=0.01+i/100
        score= f1_score(targets, (preds>thresh).astype(int))
        if score>bestscore:
            bestscore=score
            bestthresh=thresh
    return round(bestthresh,2), bestscore
        
def zeroonescale(img):
    ret=img.copy()
    ret=(ret-ret.min())/(ret.max()-ret.min())
    return ret
def zeroonescaleCH(img):
    ret=img.copy()
    for ch in range(img.shape[-1]):
        ret[...,ch]=zeroonescale(ret[...,ch])
    return ret
def robustCH(img, q05, q50, q95, channels=np.arange(12)):
    ret=img[...,channels].copy()
    for i, ch in enumerate(channels):
        ret[...,i]= (img[...,ch]-q50[ch])/(q95[ch]-q05[ch])
    return ret
def standardCH(img, allM, allS, channels=np.arange(12)):
    ret=img[...,channels].copy()
    for i, ch in enumerate(channels):
        ret[...,i]= (img[...,ch]-allM[ch])/(allS[ch])
    return ret    
def ZeroOneCH(img, allmin, allmax, channels=np.arange(12)):
    ret=img[...,channels].copy()
    for i, ch in enumerate(channels):
        ret[...,i]= (img[...,ch]-allmin[ch])/(allmax[ch]-allmin[ch])
    return ret

def create_dir(path):
    if os.path.isdir(path)==False:
        os.makedirs(path)

In [5]:
def feature_engineering(image):
    ndvi = (image[..., 3] - image[..., 0]) / (image[..., 3] + image[..., 0] + 1e-10)
    ndwi = (image[..., 3] - image[..., 1]) / (image[..., 3] + image[..., 1] + 1e-10)
    nirbleu = (image[..., 3] - image[..., 2]) / (image[..., 3] + image[..., 2] + 1e-10)

    bleugreen = (image[..., 2] - image[..., 1]) / (image[..., 2] + image[..., 1] + 1e-10)
    bleured = (image[..., 2] - image[..., 0]) / (image[..., 2] + image[..., 0] + 1e-10)
    greenred = (image[..., 1] - image[..., 0]) / (image[..., 1] + image[..., 0] + 1e-10)


    image = np.concatenate(
        [
            image,# * 2 - 1, # scale band data from -1 to 1
            np.expand_dims(ndvi+0.5, axis=-1),
            np.expand_dims(ndwi+0.5, axis=-1),
            np.expand_dims(nirbleu+0.5, axis=-1),

            np.expand_dims(bleugreen+0.5, axis=-1),
            np.expand_dims(bleured+0.5, axis=-1),
            np.expand_dims(greenred+0.5, axis=-1),           

        ],
        axis=-1,
    )
    return image


def get_feats(ar):
    # calculate statistical features from images for lighgbm and xgboost    
    ar=feature_engineering(ar)
    feats=[]
    for ch in range(ar.shape[-1]):
        feats.extend([np.mean(ar[...,ch]),np.median(ar[...,ch]),np.min(ar[...,ch]),np.max(ar[...,ch]), np.std(ar[...,ch]), skew(ar[...,ch].flatten()), kurtosis(ar[...,ch].flatten())])
    return np.array(feats)

In [6]:
def varsfromver(version):
    model_version=experimentation.loc[experimentation.Version==version]
    Model=model_version['Model'].values[0]
    ENCODER=model_version['Encoder'].values[0]
    CHANNELS=model_version['Channels'].values[0]
    # CHANNELS=[int(x) for x in model_version['Channels'].values[0].split(',')]
    # NUM_CHANNELS=len(CHANNELS)
    Scale = model_version['Scale'].values[0]
    Size  = model_version['Size'].values[0]
    Notes  = model_version['Notes'].values[0]
    
    return Model, ENCODER, CHANNELS, Scale, Size, Notes
# Model, ENCODER, CHANNELS, Scale, Size, Notes = varsfromver('v10')
# Model, ENCODER, CHANNELS, Scale, Size, Notes

In [7]:
def get_channels(CHANNELS):
    if len(CHANNELS.split(' '))==1:
       ret=[int(x) for x in CHANNELS.split(',')]
    else:
        temp=CHANNELS.split(', ')
        ret=[]
        for ss in temp:
            ret.append([int(x) for x in ss.split(',')])
    return ret
# CHANNELS=get_channels(CHANNELS)    
# CHANNELS

In [8]:
Model, ENCODER, CHANNELS, Scale, Size, Notes  = varsfromver('v10')
CHANNELS=get_channels(CHANNELS) 
Model, ENCODER, CHANNELS, Scale, Size, Notes 

('timm',
 'maxvit_rmlp_tiny_rw_256.sw_in1k',
 [0, 1, 2, 3],
 'robust 05,50,95',
 256,
 nan)

In [9]:
# modelsDICT={
#     'timm':timm.create_model(ENCODER, in_chans=NUM_CHANNELS,
#                         pretrained=True, num_classes=1)
# }
# model=modelsDICT['timm']
# model

In [10]:

validation_transformations = albumentations.Compose(
    [
        albumentations.Resize(Size, Size, interpolation=1),
    ]
)


In [11]:
# Load Train.csv and inspect the data
train_df = pd.read_csv(train_csv_path)
print("Train.csv:")
print(train_df.head())

Train.csv:
          ID  label
0  ID_HUD1ST      1
1  ID_KGE2HY      1
2  ID_VHV9BL      1
3  ID_ZT0VEJ      0
4  ID_5NFXVY      0


In [12]:
# Load all training data to ram
if CALC_OOF==True:
    folder_path=train_data_path+'/'
    X = np.array([np.load(folder_path+image_id+'.npy') for image_id in train_df['ID']])
    y = train_df['label'].values

# Load all test data to ram
test_df = pd.read_csv(test_csv_path)
test_ids = test_df['ID'].values
X_test = np.array([np.load(test_data_path+'/'+image_id+'.npy') for image_id in test_ids])

In [13]:
def create_submission(test_preds, name, THRESH=0.5):
    # 0.5 threshold
    test_preds01 = (test_preds>=THRESH).astype(int)
    
    unique, counts = np.unique(test_preds01, return_counts=True)
    prediction_counts = dict(zip(unique, counts))
    print("Prediction counts:", prediction_counts)
    
    # Prepare submission file
    submission_df = pd.DataFrame({
        'ID': test_ids,
        'label': test_preds01.flatten() })
    submission_df.to_csv(name+'THRESH'+str(THRESH)+'.csv', index=False)

In [14]:
# # CREATE STATISTCS PER CHANNELS FOR SCALING IMAGES TO FEED NNs
# Q05=np.quantile(np.concatenate((X,X_test)),q=0.05,axis=(0,1,2))
# Q50=np.quantile(np.concatenate((X,X_test)),q=0.50,axis=(0,1,2))
# Q95=np.quantile(np.concatenate((X,X_test)),q=0.95,axis=(0,1,2))

# allM=np.mean(np.concatenate((X,X_test)), axis=(0,1,2))
# allS=np.std(np.concatenate((X,X_test)), axis=(0,1,2))

# Q05,Q50,Q95,allM,allS

In [15]:
if CALC_OOF==True:
    Q05=np.quantile(np.concatenate((X,X_test)),q=0.05,axis=(0,1,2))
    Q50=np.quantile(np.concatenate((X,X_test)),q=0.50,axis=(0,1,2))
    Q95=np.quantile(np.concatenate((X,X_test)),q=0.95,axis=(0,1,2))
    
    allM=np.mean(np.concatenate((X,X_test)), axis=(0,1,2))
    allS=np.std(np.concatenate((X,X_test)), axis=(0,1,2))
else:
    Q05  =np.array([ 1054.        , 1141.        , 1108.        , 1432.        ,
                     -23.84737916,  -43.12207857,   -7.87792338,  -11.21783106,
                     -20.88170549,  -40.68729514,   -3.84420443,   -7.63233113])
    Q50  =np.array([ 1.51100000e+03,  1.65600000e+03,  1.56000000e+03,  2.97500000e+03,
                    -1.02171109e+01, -1.65654301e+01, -2.84631043e-01, -7.57945780e-02,
                    -1.14755997e+01, -1.77631617e+01,  3.88509645e-01, -6.33488874e-02])
    Q95  =np.array([ 4.00000000e+03,  3.94200000e+03,  3.86400000e+03,  5.50800000e+03,
                     2.51268458e+00, -5.03906754e+00,  3.69729500e+00,  5.10263839e+00,
                    -3.37361028e-02, -7.10041075e+00,  5.82530082e+00,  6.01170365e+00])
    allM =np.array([ 1.85185963e+03,  1.92838193e+03,  1.86725082e+03,  3.11747824e+03,
                    -1.05585540e+01, -1.88282667e+01, -9.60078930e-01, -7.49858269e-01,
                    -1.13484484e+01, -2.01162665e+01,  5.81196265e-01, -2.75804596e-01])
    allS =np.array([1341.69370135, 1273.92620502, 1278.76344252, 1466.91812707,
                   8.34187043,   10.65859149,    4.33202752,    5.08455682,
                   6.43914294,    9.9298614 ,    2.93451315,    4.50931721])

In [16]:

num_channels=feature_engineering(X_test[0]).shape[-1]
colnames=[['meanCH'+str(ch),'medianCH'+str(ch),'minCH'+str(ch),'maxCH'+str(ch),'stdCH'+str(ch),'skewCH'+str(ch),'kurtosisCH'+str(ch)] for ch in range(num_channels)]
colnames=np.hstack(colnames)
colnames.shape

print('Create Statistical features datasets from images, for LighGBM and XGBoost models')
if CALC_OOF==True:
    Xfeats=np.stack([get_feats(X[i]) for i in tqdm(range(len(X)))])
    Xfeats.shape
Xtestfeats=np.stack([get_feats(X_test[i]) for i in tqdm(range(len(X_test)))])
Xtestfeats.shape

Create Statistical features datasets from images, for LighGBM and XGBoost models


100%|██████████| 5398/5398 [02:25<00:00, 37.08it/s]


(5398, 126)

In [17]:
# LightGBM
name='Landslide_Class_S2S1_v18b'

if CALC_OOF==True:
    OOF=np.zeros(len(y))
    val_true=np.zeros(len(X))

testpreds=np.zeros(len(X_test))

if CALC_OOF==True:
    sgkf = KFold(n_splits=N_folds, random_state=42, shuffle=True)
    for fold, (train_index, test_index) in enumerate(sgkf.split(y, y)):
        model=joblib.load(TRAINED_MODELS_PATH+name+'_fold'+str(fold))   
   
        preds = model.predict_proba(Xfeats[test_index])[:,1]  
        OOF[test_index]=preds
        val_true[test_index]=y[test_index]
        F1 = f1_score(y[test_index], (preds>THRESH).astype(int))
        print(F1)
        gc.collect()
else:
    for fold in range(N_folds):
        model=joblib.load(TRAINED_MODELS_PATH+name+'_fold'+str(fold)) 
        testpreds += model.predict_proba(Xtestfeats)[:,1]/N_folds
    
        gc.collect()

if CALC_OOF==True:
    np.savez( 'OOF_'+name, OOF)
    print(f1_score(y, (OOF>THRESH).astype(int)), find_opt_thresh(y, OOF))

np.savez( 'testProbs_'+name, testpreds)
create_submission(test_preds=testpreds, name=name, THRESH=0.5)

/usr/local/lib/python3.11/dist-packages/sklearn/base.py:318: UserWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:318: UserWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:318: UserWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more inf

Prediction counts: {0: 4779, 1: 619}


In [18]:
# CALC_OOF=True

In [19]:
# XGBoost
name='Landslide_Class_S2S1_v44'

if CALC_OOF==True:
    OOF=np.zeros(len(y))
    val_true=np.zeros(len(X))

testpreds=np.zeros(len(X_test))

if CALC_OOF==True:
    sgkf = KFold(n_splits=N_folds, random_state=42, shuffle=True)
    for fold, (train_index, test_index) in enumerate(sgkf.split(y, y)):
        model=joblib.load(TRAINED_MODELS_PATH+name+'_fold'+str(fold))   
   
        preds = model.predict_proba(Xfeats[test_index])[:,1]  
        OOF[test_index]=preds
        val_true[test_index]=y[test_index]
        F1 = f1_score(y[test_index], (preds>THRESH).astype(int))
        print(F1)
        gc.collect()
else:
    for fold in range(N_folds):
        model=joblib.load(TRAINED_MODELS_PATH+name+'_fold'+str(fold)) 
        testpreds += model.predict_proba(Xtestfeats)[:,1]/N_folds
    
        gc.collect()

if CALC_OOF==True:
    np.savez( 'OOF_'+name, OOF)
    print(f1_score(y, (OOF>THRESH).astype(int)), find_opt_thresh(y, OOF))

np.savez( 'testProbs_'+name, testpreds)
create_submission(test_preds=testpreds, name=name, THRESH=0.5)

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [12:23:24] WARNING: /workspace/src/common/error_msg.h:80: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  warnings.warn(smsg, UserWarning)


Prediction counts: {0: 4775, 1: 623}


In [20]:
# get_all_model_params(model_version),model_version
# v31,v33-->combinemodels2a
# v37,v39-->combinemodels3a
# v46-->combinemodels3c
class combinemodels2a(nn.Module):
    def __init__(self):
        super(combinemodels2a, self).__init__()

        self.model1    = timm.create_model(ENCODER, in_chans=4,
                                                        pretrained=False, num_classes=256)
        self.model2    = timm.create_model(ENCODER, in_chans=4,
                                            pretrained=False, num_classes=256)

      
        self.feats = nn.Linear(in_features=256+256, out_features=256)
        self.bn = nn.BatchNorm1d(256)
        self.clf = nn.Linear(256*3, 1)



    def forward(self, x):
        # Concatenate along the time dimension
        rgbn = self.model1(x[:,:4,...])
        sar = self.model2(x[:,-4:,...])
        combined = torch.cat([rgbn, sar], 1)

        reducedfeats = self.feats(combined)
        reducedfeats = self.bn(reducedfeats)
        combined2 = torch.cat([combined, reducedfeats], 1)
        output = self.clf(combined2)

        return output

class combinemodels3a(nn.Module):
    def __init__(self):
        super(combinemodels3a, self).__init__()

        self.model1    = timm.create_model(ENCODER, in_chans=4,
                                                        pretrained=False, num_classes=256)
        self.model2    = timm.create_model(ENCODER, in_chans=4,
                                            pretrained=False, num_classes=256)
        self.model3    = timm.create_model(ENCODER, in_chans=6,
                                            pretrained=False, num_classes=256)

        self.feats = nn.Linear(in_features=256+256+256, out_features=256+256)
        self.bn = nn.BatchNorm1d(256+256)
        self.comb = nn.Linear(256*5, 256*2)

        self.do = nn.Dropout(p=0.1, inplace=False)
        self.clf = nn.Linear(256*2, 1)


    def forward(self, x):
        rgbn = self.model1(x[:,:4,...])
        sarDiff = self.model2(x[:,4:8,...])
        rgbnFs = self.model3(x[:,8:,...])
        combined = torch.cat([rgbn, sarDiff, rgbnFs], 1)

        reducedfeats = self.feats(combined)
        reducedfeats = self.bn(reducedfeats)
        combined2 = torch.cat([combined, reducedfeats], 1)
        comb = self.comb(combined2)
        comb = self.do(comb)
        output = self.clf(comb)
        return output
        
class combinemodels3c(nn.Module):
    def __init__(self):
        super(combinemodels3c, self).__init__()

        self.model1    = timm.create_model(ENCODER, in_chans=4,
                                                        pretrained=False, num_classes=256)
        self.model2    = timm.create_model(ENCODER, in_chans=4,
                                            pretrained=False, num_classes=256)
        self.model3    = timm.create_model(ENCODER, in_chans=4,
                                            pretrained=False, num_classes=256)

        self.feats = nn.Linear(in_features=256+256+256, out_features=256)
        self.act = nn.ReLU()
        self.clf = nn.Linear(256*4, 1)

    def forward(self, x):
        rgbn = self.model1(x[:,:4,...])
        sar = self.model2(x[:,4:8,...])
        sarDiff = self.model3(x[:,8:,...])
        combined = torch.cat([rgbn, sar, sarDiff], 1)

        reducedfeats = self.feats(combined)
        reducedfeats = self.act(reducedfeats)
        combined2 = torch.cat([combined, reducedfeats], 1)
        output = self.clf(combined2)

        return output

In [21]:
def get_all_model_params(version):
    Model, ENCODER, CHANNELS, Scale, Size, Notes = varsfromver(version)
    # Model, ENCODER, CHANNELS, Scale, Size, Notes
    CHANNELS=get_channels(CHANNELS) 
    # weights_path='/kaggle/input/'+findweightset4modelversion(version=version)+'/'
    weights_path=TRAINED_MODELS_PATH
    name='Landslide_Class_S2S1_'+version
    if 'from optical bands' in str(Notes):
        FEAT_ENG='fromOpticalBands'
    # elif '4diffRGBNnSAR' in str(Notes):
    #     FEAT_ENG='4diffRGBNnSAR'
    else:
        FEAT_ENG=None
    return Model, ENCODER, CHANNELS, Scale, Size, FEAT_ENG, weights_path, name


In [22]:
model_version='v10'
Model, ENCODER, CHANNELS, Scale, Size, FEAT_ENG, weights_path, name=get_all_model_params(model_version)
Model, ENCODER, CHANNELS, Scale, Size, FEAT_ENG, weights_path, name

('timm',
 'maxvit_rmlp_tiny_rw_256.sw_in1k',
 [0, 1, 2, 3],
 'robust 05,50,95',
 256,
 None,
 '/kaggle/input/landslide-class-weights/',
 'Landslide_Class_S2S1_v10')

In [23]:
def select_model(Model, ENCODER, CHANNELS):
    # v31,v33-->combinemodels2a
    # v37,v39-->combinemodels3a
    # v46-->combinemodels3c
    if Model=='timm':
        model = timm.create_model(ENCODER, in_chans=len(CHANNELS), pretrained=False, num_classes=1)
        
    elif Model=='combinemodelsV2a':
        model=combinemodels2a()
    elif Model=='combinemodelsV3a':
        model=combinemodels3a()
    elif Model=='combinemodelsV3c':
        model=combinemodels3c()
        
    model.to('cuda')
    model.eval()
    print('model created')
    
    return model
model = select_model(Model, ENCODER, CHANNELS)

model created


In [24]:
class Dataset_ram(torch.utils.data.Dataset):

    def __init__(self, iminds, transforms=None, channels=np.arange(12), Scale='robust 05,50,95', feat_eng=None, mode='train'):
        self.iminds = iminds
        self.transforms = transforms
        self.channels=channels
        self.Scale=Scale
        self.feat_eng=feat_eng
        self.mode=mode

    def __len__(self):
        return len(self.iminds)

    def __getitem__(self, idx):
        # Loads a 3-channel image from a chip-level dataframe


#         x_arr=np.clip(x_arr,CLIPMIN,CLIPMAX)

        if self.mode=='test':
            y_arr=np.zeros((256,256)).astype('uint8')
            x_arr=X_test[self.iminds[idx]]
        else:
            y_arr=y[self.iminds[idx]]
            x_arr=X[self.iminds[idx]]

        
        #the 2 main scalings used are standardScale and robust 05,50,95
        if self.Scale=='standardScale':
            x_arr = standardCH(x_arr.astype('float32'), allM=allM, allS=allS)
        elif self.Scale=='robust 05,50,95':
            x_arr = robustCH(x_arr.astype('float32'), q05=Q05, q50=Q50, q95=Q95)    
        else:
            print('other scaling')

        
        if self.feat_eng=='fromOpticalBands':
            x_arr=feature_engineering(x_arr)
        
        x_arr=x_arr[...,self.channels]

        y_arr = np.expand_dims(y_arr, -1)#.astype('float32')

        # Apply data augmentations, if provided
        if self.transforms:
            augmented = self.transforms(image=x_arr)
            x_arr = augmented['image']
      

        x_arr = np.transpose(x_arr, [2, 0, 1])


        return x_arr.astype('float32'), y_arr#.astype('float32')#.astype('uint8')#, not_missing_mask.astype('uint8')

def get_test_preds(channels, SCALE, FEAT_ENG, weights_path, name):

    test_dataset = Dataset_ram(iminds=np.arange(len(X_test)),
                                         transforms = validation_transformations,
                                         channels=channels, Scale=SCALE, feat_eng=FEAT_ENG,
                                         mode='test'
                                        )
    test_loader = DataLoader(test_dataset, batch_size=BS, shuffle=False, num_workers=4)
    
    all_test_preds= []
    for fold in range(N_folds):
    
        model.load_state_dict(torch.load(weights_path+name+'_Fold'+str(fold)+'_last.pt'))
        # model.load_state_dict(torch.load(weights_path+name+'_Fold'+str(fold)+'_last.pt', map_location=torch.device('cpu')))
        test_preds= []
    
        with torch.no_grad():
            pb = progress_bar(test_loader)
            for i, (x_batch, y_batch) in enumerate(pb):
    
                preds0 = model(x_batch.cuda(non_blocking=True)).detach()
                preds1 = model(torch.flip(x_batch, dims=[2]).cuda(non_blocking=True)).detach()
                preds2 = model(torch.flip(x_batch, dims=[3]).cuda(non_blocking=True)).detach()
                preds3 = model(torch.flip(x_batch, dims=[2,3]).cuda(non_blocking=True)).detach()
    
                preds=(preds0 + preds1 + preds2 + preds3)/4
    
                preds=torch.sigmoid(preds)
    
                test_preds.append(preds.cpu())
    
    
        test_preds=np.vstack(test_preds)[:,0]
        all_test_preds.append(test_preds)
    test_preds=np.vstack(all_test_preds)

    test_preds=np.mean(test_preds,0)
    return test_preds


def get_OOF(weights_path, name, channels, SCALE, FEAT_ENG):
    # sgkf = KFold(n_splits=N_folds, random_state=42, shuffle=True)
    OOF=np.zeros(len(y))
    for fold, (train_index, test_index) in enumerate(sgkf.split(y, y)):
    
        model.load_state_dict(torch.load(weights_path+name+'_Fold'+str(fold)+'_last.pt'))
    
        # model.cuda()
    
        valid_dataset = Dataset_ram(iminds=test_index,
                                         transforms = validation_transformations,
                                         channels=channels, Scale=SCALE, feat_eng=FEAT_ENG,
                                         mode='eval'
                                        )
    
        valid_loader = DataLoader(valid_dataset, batch_size=BS, shuffle=False, num_workers=num_workers)
    
        criterion0 =  nn.BCEWithLogitsLoss()
        criterion = SmoothF1Loss()
    
        # best_score=0.
    
    
        # model.eval()
        avg_val_loss = 0.
    
        val_preds, val_true =[], []
    
        with torch.no_grad():
            pb = progress_bar(valid_loader)
            for i, (x_batch, y_batch) in enumerate(pb):
                x_batch=x_batch.cuda(non_blocking=True)
                y_batch=y_batch.cuda(non_blocking=True)
    
                preds = model(x_batch)
    
                loss = criterion(preds, y_batch.to(dtype=torch.float))
    
                preds=torch.sigmoid(preds)
                val_preds.append(preds.cpu())
                val_true.append(y_batch.cpu())
    
    
                avg_val_loss += loss.item() / len(valid_loader)
    
    
            print('Val Loss: {:.6f}'.format(avg_val_loss))
    
            val_preds=np.vstack(val_preds)[:,0]
            val_true=np.vstack(val_true)[:,0]
    
            F1 = f1_score(val_true, (val_preds>THRESH).astype(int))
            print(F1)
    
            # adjust_optim(optimizer, (epoch + 1))
    
            gc.collect()
    
        OOF[test_index]=val_preds
    return OOF



In [25]:
if CALC_OOF==True:
    OOF = get_OOF(weights_path, name, channels=CHANNELS, SCALE=Scale, FEAT_ENG=FEAT_ENG)    
    np.savez( 'OOF_'+name, OOF)
    bestthresh, bestscore=find_opt_thresh(y, OOF)
    f1_score(y, (OOF>THRESH).astype(int)), bestthresh, bestscore

In [26]:
test_preds=get_test_preds(channels=CHANNELS, SCALE=Scale, FEAT_ENG=FEAT_ENG, weights_path=weights_path, name=name)

np.savez( 'testProbs_'+name, test_preds)

create_submission(test_preds=test_preds, name=name, THRESH=0.5)

Prediction counts: {0: 4823, 1: 575}


In [27]:
model_version='v20'
Model, ENCODER, CHANNELS, Scale, Size, FEAT_ENG, weights_path, name=get_all_model_params(model_version)
print(Model, ENCODER, CHANNELS, Scale, Size, FEAT_ENG, weights_path, name)
model = select_model(Model, ENCODER, CHANNELS)

timm vit_medium_patch16_reg4_gap_256.sbb_in12k_ft_in1k [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17] robust 05,50,95 256 fromOpticalBands /kaggle/input/landslide-class-weights/ Landslide_Class_S2S1_v20
model created


In [28]:
if CALC_OOF==True:
    OOF = get_OOF(weights_path, name, channels=CHANNELS, SCALE=Scale, FEAT_ENG=FEAT_ENG)    
    np.savez( 'OOF_'+name, OOF)
    bestthresh, bestscore=find_opt_thresh(y, OOF)
    f1_score(y, (OOF>THRESH).astype(int)), bestthresh, bestscore

In [29]:
test_preds=get_test_preds(channels=CHANNELS, SCALE=Scale, FEAT_ENG=FEAT_ENG, weights_path=weights_path, name=name)
np.savez( 'testProbs_'+name, test_preds)
create_submission(test_preds=test_preds, name=name, THRESH=0.5)

Prediction counts: {0: 4763, 1: 635}


In [30]:
model_version='v31'
Model, ENCODER, CHANNELS, Scale, Size, FEAT_ENG, weights_path, name=get_all_model_params(model_version)
print(Model, ENCODER, CHANNELS, Scale, Size, FEAT_ENG, weights_path, name)
model = select_model(Model, ENCODER, CHANNELS)

combinemodelsV2a vit_pwee_patch16_reg1_gap_256.sbb_in1k [[0, 1, 2, 3], [6, 7, 10, 11]] standardScale 256 None /kaggle/input/landslide-class-weights/ Landslide_Class_S2S1_v31
model created


In [31]:
if CALC_OOF==True:
    OOF = get_OOF(weights_path, name, channels=CHANNELS[0]+CHANNELS[1], SCALE=Scale, FEAT_ENG=FEAT_ENG)    
    np.savez( 'OOF_'+name, OOF)
    bestthresh, bestscore=find_opt_thresh(y, OOF)
    f1_score(y, (OOF>THRESH).astype(int)), bestthresh, bestscore

In [32]:
test_preds=get_test_preds(CHANNELS[0]+CHANNELS[1], SCALE=Scale, FEAT_ENG=FEAT_ENG, weights_path=weights_path, name=name)

np.savez( 'testProbs_'+name, test_preds)
create_submission(test_preds=test_preds, name=name, THRESH=0.5)

Prediction counts: {0: 4772, 1: 626}


In [33]:
model_version='v33'
Model, ENCODER, CHANNELS, Scale, Size, FEAT_ENG, weights_path, name=get_all_model_params(model_version)
print(Model, ENCODER, CHANNELS, Scale, Size, FEAT_ENG, weights_path, name)
model = select_model(Model, ENCODER, CHANNELS)

if CALC_OOF==True:
    OOF = get_OOF(weights_path, name, channels=CHANNELS[0]+CHANNELS[1], SCALE=Scale, FEAT_ENG=FEAT_ENG)    
    np.savez( 'OOF_'+name, OOF)
    bestthresh, bestscore=find_opt_thresh(y, OOF)
    print(f1_score(y, (OOF>THRESH).astype(int)), bestthresh, bestscore)


test_preds=get_test_preds(channels=CHANNELS[0]+CHANNELS[1], SCALE=Scale, FEAT_ENG=FEAT_ENG, weights_path=weights_path, name=name)
np.savez( 'testProbs_'+name, test_preds)
create_submission(test_preds=test_preds, name=name, THRESH=0.5)

combinemodelsV2a vit_pwee_patch16_reg1_gap_256.sbb_in1k [[0, 1, 2, 3], [6, 7, 10, 11]] robust 05,50,95 256 None /kaggle/input/landslide-class-weights/ Landslide_Class_S2S1_v33
model created


Prediction counts: {0: 4775, 1: 623}


In [34]:
model_version='v37'
Model, ENCODER, CHANNELS, Scale, Size, FEAT_ENG, weights_path, name=get_all_model_params(model_version)
print(Model, ENCODER, CHANNELS, Scale, Size, FEAT_ENG, weights_path, name)
model = select_model(Model, ENCODER, CHANNELS)

if CALC_OOF==True:
    OOF = get_OOF(weights_path, name, channels=CHANNELS[0]+CHANNELS[1]+CHANNELS[2], SCALE=Scale, FEAT_ENG=FEAT_ENG)    
    np.savez( 'OOF_'+name, OOF)
    bestthresh, bestscore=find_opt_thresh(y, OOF)
    print(f1_score(y, (OOF>THRESH).astype(int)), bestthresh, bestscore)


test_preds=get_test_preds(channels=CHANNELS[0]+CHANNELS[1]+CHANNELS[2], SCALE=Scale, FEAT_ENG=FEAT_ENG, weights_path=weights_path, name=name)
np.savez( 'testProbs_'+name, test_preds)
create_submission(test_preds=test_preds, name=name, THRESH=0.5)

combinemodelsV3a vit_pwee_patch16_reg1_gap_256.sbb_in1k [[0, 1, 2, 3], [6, 7, 10, 11], [12, 13, 14, 15, 16, 17]] standardScale 256 fromOpticalBands /kaggle/input/landslide-class-weights/ Landslide_Class_S2S1_v37
model created


Prediction counts: {0: 4775, 1: 623}


In [35]:
model_version='v39'
Model, ENCODER, CHANNELS, Scale, Size, FEAT_ENG, weights_path, name=get_all_model_params(model_version)
print(Model, ENCODER, CHANNELS, Scale, Size, FEAT_ENG, weights_path, name)
model = select_model(Model, ENCODER, CHANNELS)

if CALC_OOF==True:
    OOF = get_OOF(weights_path, name, channels=CHANNELS[0]+CHANNELS[1]+CHANNELS[2], SCALE=Scale, FEAT_ENG=FEAT_ENG)    
    np.savez( 'OOF_'+name, OOF)
    bestthresh, bestscore=find_opt_thresh(y, OOF)
    print(f1_score(y, (OOF>THRESH).astype(int)), bestthresh, bestscore)


test_preds=get_test_preds(channels=CHANNELS[0]+CHANNELS[1]+CHANNELS[2], SCALE=Scale, FEAT_ENG=FEAT_ENG, weights_path=weights_path, name=name)
np.savez( 'testProbs_'+name, test_preds)
create_submission(test_preds=test_preds, name=name, THRESH=0.5)

combinemodelsV3a caformer_s18.sail_in22k_ft_in1k [[0, 1, 2, 3], [6, 7, 10, 11], [12, 13, 14, 15, 16, 17]] standardScale 256 fromOpticalBands /kaggle/input/landslide-class-weights/ Landslide_Class_S2S1_v39
model created


Prediction counts: {0: 4782, 1: 616}


In [36]:
model_version='v46'
Model, ENCODER, CHANNELS, Scale, Size, FEAT_ENG, weights_path, name=get_all_model_params(model_version)
print(Model, ENCODER, CHANNELS, Scale, Size, FEAT_ENG, weights_path, name)
model = select_model(Model, ENCODER, CHANNELS)

if CALC_OOF==True:
    OOF = get_OOF(weights_path, name, channels=CHANNELS[0]+CHANNELS[1]+CHANNELS[2], SCALE=Scale, FEAT_ENG=FEAT_ENG)    
    np.savez( 'OOF_'+name, OOF)
    bestthresh, bestscore=find_opt_thresh(y, OOF)
    print(f1_score(y, (OOF>THRESH).astype(int)), bestthresh, bestscore)

test_preds=get_test_preds(channels=CHANNELS[0]+CHANNELS[1]+CHANNELS[2], SCALE=Scale, FEAT_ENG=FEAT_ENG, weights_path=weights_path, name=name)
np.savez( 'testProbs_'+name, test_preds)
create_submission(test_preds=test_preds, name=name, THRESH=0.5)

combinemodelsV3c vit_medium_patch16_rope_reg1_gap_256.sbb_in1k [[0, 1, 2, 3], [4, 5, 8, 9], [6, 7, 10, 11]] standardScale 256 None /kaggle/input/landslide-class-weights/ Landslide_Class_S2S1_v46
model created


Prediction counts: {0: 4772, 1: 626}


In [37]:
names=[
            'Landslide_Class_S2S1_v18b',
            'Landslide_Class_S2S1_v10',
            'Landslide_Class_S2S1_v20',
            'Landslide_Class_S2S1_v31',
            'Landslide_Class_S2S1_v33',
            'Landslide_Class_S2S1_v37',
            'Landslide_Class_S2S1_v39',
            'Landslide_Class_S2S1_v44',
            'Landslide_Class_S2S1_v46',
            # 'Landslide_Class_S2S1_v36',
            # 'Landslide_Class_S2S1_v49',
    ]

if CALC_OOF==True:
    
    all_OOFs=[]
    for i, name in enumerate(names):
        v=np.load('OOF_'+name+'.npz')['arr_0']
        all_OOFs.append(v)
    all_OOFs= np.stack(all_OOFs)   
    all_OOFs=np.mean(all_OOFs,0)
    
    bestthresh, bestscore=find_opt_thresh(y, all_OOFs)
    print(THRESH,f1_score(y, (all_OOFs>THRESH).astype(int)), bestthresh, bestscore)

In [38]:
# names=[
#     'Landslide_Class_S2S1_v18b',
#     'Landslide_Class_S2S1_v10',
#     'Landslide_Class_S2S1_v20',
#     'Landslide_Class_S2S1_v31',
#     'Landslide_Class_S2S1_v33',
#     'Landslide_Class_S2S1_v37',
#     'Landslide_Class_S2S1_v39',
#     'Landslide_Class_S2S1_v44',
#     'Landslide_Class_S2S1_v46',
#     # 'Landslide_Class_S2S1_v36',
#     # 'Landslide_Class_S2S1_v49',
#     ]
all_test_probs=[]
for i, name in enumerate(names):
    v=np.load('testProbs_'+name+'.npz')['arr_0']
    all_test_probs.append(v)
for i in range(1,5):
    print(i,np.corrcoef(all_test_probs[0],all_test_probs[i])[0,1])    
all_test_probs= np.stack(all_test_probs)   
all_test_probs=np.mean(all_test_probs,0)

1 0.8902968145877688
2 0.8957622676235603
3 0.9055688875129971
4 0.9097624947499483


In [39]:
# 0.5 threshold
test_preds01 = (all_test_probs>0.5).astype(int)

unique, counts = np.unique(test_preds01, return_counts=True)
prediction_counts = dict(zip(unique, counts))
print("Prediction counts:", prediction_counts)

# Prepare submission file
submission_df = pd.DataFrame({
    'ID': test_ids,
    'label': test_preds01.flatten() })
submission_df.to_csv('AVG5THRESH'+str(0.5)+'.csv', index=False)

Prediction counts: {0: 4785, 1: 613}


In [40]:
# 0.5 threshold
test_preds01 = (all_test_probs>bestthresh).astype(int)

unique, counts = np.unique(test_preds01, return_counts=True)
prediction_counts = dict(zip(unique, counts))
print("Prediction counts:", prediction_counts)

# Prepare submission file
submission_df = pd.DataFrame({
    'ID': test_ids,
    'label': test_preds01.flatten() })
submission_df.to_csv('AVG5THRESH'+str(bestthresh)+'.csv', index=False)

Prediction counts: {0: 4782, 1: 616}
